In [11]:
import numpy as np
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display, Markdown

# Constants
mu_0 = 4 * np.pi * 1e-7
epsilon_0 = 8.854187817e-12
c = 1 / np.sqrt(mu_0 * epsilon_0)

def plot_waveguide(mode_type="TE", field_type="Electric Field |E|", m=1, n=0, freq_GHz=10.0, a=1.0, b=0.5, L=5.0):
    # Create spatial grid
    Nx, Ny, Nz = 30, 15, 60
    x = np.linspace(0, a, Nx)
    y = np.linspace(0, b, Ny)
    z = np.linspace(0, L, Nz)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

    freq = freq_GHz * 1e9
    omega = 2 * np.pi * freq
    k = omega / c
    kc2 = (m * np.pi / a)**2 + (n * np.pi / b)**2
    fc = np.sqrt(kc2) * c / (2 * np.pi)

    if m == 0 and n == 0:
        display(Markdown("⚠️ **TE₀₀/TM₀₀ mode does not exist.**"))
        return
    if mode_type == "TM" and (m == 0 or n == 0):
        display(Markdown("⚠️ **TM₀₀, TM₀n, and TMm₀ modes do not exist.**"))
        return
    if k**2 <= kc2:
        display(Markdown(f"⚠️ **Below cutoff:** f = {freq_GHz:.2f} GHz < f_c = {fc/1e9:.2f} GHz. Mode will not propagate."))
        return

    beta_mn = np.sqrt(k**2 - kc2)
    neff = beta_mn * c / omega

    # Display key values
    display(Markdown(f"""
### 📐 Propagation Parameters
- **Cutoff Frequency:** {fc / 1e9:.2f} GHz  
- **Phase Constant (βₘₙ):** {beta_mn:.2f} 1/m  
- **Effective Refractive Index (nₑff):** {neff:.4f}
    """))

    # Calculate Fields
    if mode_type == "TE":
        if field_type == "Electric Field |E|":
            Ex = (1j * omega * mu_0 * n * np.pi / (kc2 * b)) * np.cos(m * np.pi * X / a) * np.sin(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            Ey = -(1j * omega * mu_0 * m * np.pi / (kc2 * a)) * np.sin(m * np.pi * X / a) * np.cos(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            E_mag = np.abs(Ex)**2 + np.abs(Ey)**2
            data = np.sqrt(E_mag)
            colorbar_title = '|E| (TE)'
            colorscale = 'Viridis'
        else:
            Hx = (1j * beta_mn * m * np.pi / (kc2 * a)) * np.sin(m * np.pi * X / a) * np.cos(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            Hy = (1j * beta_mn * n * np.pi / (kc2 * b)) * np.cos(m * np.pi * X / a) * np.sin(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            Hz = np.cos(m * np.pi * X / a) * np.cos(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            H_mag = np.abs(Hx)**2 + np.abs(Hy)**2 + np.abs(Hz)**2
            data = np.sqrt(H_mag)
            colorbar_title = '|H| (TE)'
            colorscale = 'Blues'
    else:
        if field_type == "Electric Field |E|":
            Ez = np.sin(m * np.pi * X / a) * np.sin(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            Ex = (1j * beta_mn * m * np.pi / (kc2 * a)) * np.cos(m * np.pi * X / a) * np.sin(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            Ey = (1j * beta_mn * n * np.pi / (kc2 * b)) * np.sin(m * np.pi * X / a) * np.cos(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            E_mag = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
            data = np.sqrt(E_mag)
            colorbar_title = '|E| (TM)'
            colorscale = 'Cividis'
        else:
            Hx = (1j * omega * epsilon_0 * n * np.pi / (kc2 * b)) * np.cos(m * np.pi * X / a) * np.sin(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            Hy = (1j * omega * epsilon_0 * m * np.pi / (kc2 * a)) * np.sin(m * np.pi * X / a) * np.cos(n * np.pi * Y / b) * np.exp(-1j * beta_mn * Z)
            H_mag = np.abs(Hx)**2 + np.abs(Hy)**2
            data = np.sqrt(H_mag)
            colorbar_title = '|H| (TM)'
            colorscale = 'Oranges'

    # Normalize and Plot Field
    fig1 = go.Figure(data=go.Scatter3d(
        x=Z.flatten(), y=X.flatten(), z=Y.flatten(),
        mode='markers',
        marker=dict(
            size=2.5,
            symbol='square',
            color=(data / np.max(data)).flatten(),
            colorscale=colorscale,
            opacity=1.0,
            colorbar=dict(title=colorbar_title)
        )
    ))

    fig1.update_layout(
        title=f"{mode_type} {field_type} (m={m}, n={n}, f={freq_GHz:.2f} GHz)",
        scene=dict(
            xaxis=dict(title="z (propagation)", range=[0, L]),
            yaxis=dict(title="x (width)", range=[0, a]),
            zaxis=dict(title="y (height)", range=[0, b]),
            aspectmode='manual',
            aspectratio=dict(x=L, y=a, z=b),
            camera=dict(eye=dict(x=1.4, y=1.4, z=0.6))
        ),
        width=850,
        height=650
    )
    fig1.show()

    # Phase constant and neff graphs
    f_range = np.linspace(0.1e9, 15e9, 500)
    k_range = 2 * np.pi * f_range / c
    beta_range = np.sqrt(np.maximum(0, k_range**2 - kc2))
    neff_range = beta_range * c / (2 * np.pi * f_range)

    fig_beta = go.Figure()
    fig_beta.add_trace(go.Scatter(x=f_range / 1e9, y=beta_range, mode='lines', name='βₘₙ', line=dict(color='blue')))
    fig_beta.add_vline(x=fc / 1e9, line=dict(color='red', dash='dot'), annotation_text="Cutoff", annotation_position="top right")
    fig_beta.update_layout(title="βₘₙ vs Frequency", xaxis_title="Frequency (GHz)", yaxis_title="Phase Constant (1/m)", height=400)
    fig_beta.show()

    fig_neff = go.Figure()
    fig_neff.add_trace(go.Scatter(x=f_range / 1e9, y=neff_range, mode='lines', name='nₑff', line=dict(color='green')))
    fig_neff.add_vline(x=fc / 1e9, line=dict(color='red', dash='dot'), annotation_text="Cutoff", annotation_position="top right")
    fig_neff.update_layout(title="nₑff vs Frequency", xaxis_title="Frequency (GHz)", yaxis_title="Effective Refractive Index", height=400)
    fig_neff.show()

# Widget-based UI
widgets.interact(
    plot_waveguide,
    mode_type=widgets.Dropdown(options=["TE", "TM"], value="TE", description='Mode'),
    field_type=widgets.Dropdown(options=["Electric Field |E|", "Magnetic Field |H|"], value="Electric Field |E|", description='Field'),
    m=widgets.IntSlider(min=0, max=4, value=1, step=1, description='m'),
    n=widgets.IntSlider(min=0, max=4, value=0, step=1, description='n'),
    freq_GHz=widgets.FloatSlider(min=0.1, max=15, step=0.1, value=10, description='f (GHz)'),
    a=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='a (width)'),
    b=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=0.5, description='b (height)'),
    L=widgets.FloatSlider(min=1.0, max=10.0, step=0.5, value=5.0, description='L (length)')
)


interactive(children=(Dropdown(description='Mode', options=('TE', 'TM'), value='TE'), Dropdown(description='Fi…

<function __main__.plot_waveguide(mode_type='TE', field_type='Electric Field |E|', m=1, n=0, freq_GHz=10.0, a=1.0, b=0.5, L=5.0)>